In [2]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
ROOT_DIR = Path("/mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome")
INPUT_DIR = ROOT_DIR / "input_data/HBN"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
output_dir = ROOT_DIR / "output_data" / "cleaned" / "HBN"
output_dir.mkdir(parents=True, exist_ok=True)

In [4]:
general_ses = pd.read_csv(INPUT_DIR / "HBN_General_SES.csv")
print(general_ses["GUID"].nunique())
general_ses.head(2)

6220


,GUID,General_SES
0,NDARAA066GKK,3.18
1,NDARAA075AMK,1.10


In [ ]:
# folder containing noddi data from cubic
# generated with: bash \
# 	unzip_files.sh \
# 	/cbica/projects/pennlinc_rbc/datasets/LINC_HBN/derivatives/QSIRECON-1-1-0_BUNDLE-STATS_zipped \
# 	/cbica/projects/macedo_long_wm/data/noddi \
# 	"qsirecon/derivatives/qsirecon-wmNODDI/sub-*/ses-1/dwi/sub-*_bundles-DSIStudio_scalarstats*"

noddi_dir = INPUT_DIR / "noddi"

tsv_files = sorted(noddi_dir.glob("sub-*_ses-1_acq-64dir_space-ACPC_bundles-DSIStudio_scalarstats.tsv"))
allowed_ids = set("sub-" + general_ses["GUID"].astype(str)) 

dfs = []
keep_metrics = {"icvf", "isovf", "od"}

for f in tsv_files:
    df = pd.read_csv(f, sep="\t")
    df_filt = df[df["variable_name"].isin(keep_metrics)].copy()
    df_filt["metric"] = "NODDI_" + df_filt["variable_name"].astype(str)
    df_filt = df_filt[["bundle", "metric", "mean", "subject_id"]] 
    df_filt = df_filt[df_filt["subject_id"].isin(allowed_ids)]
    dfs.append(df_filt)

# Concatenate all subjects into one DataFrame
all_noddi = pd.concat(dfs, ignore_index=True)

# save to cleaned/HBN

output_path = output_dir / "noddi_data.csv"
all_noddi.to_csv(output_path, index=False)

print(f"[SAVE] → {output_path} | shape: {all_noddi.shape}")
all_noddi.head()

In [ ]:
all_noddi["subject_id"].nunique()

In [ ]:
# folder containing macrostructure data from cubic
# generated with: bash \
# 	unzip_files.sh \
# 	/cbica/projects/pennlinc_rbc/datasets/LINC_HBN/derivatives/QSIRECON-1-1-0_BUNDLE-STATS_zipped \
# 	/cbica/projects/macedo_long_wm/data/macro \
# 	"qsirecon/derivatives/qsirecon-MSMTAutoTrack/sub-*/ses-1/dwi/sub-*msmt_bundlestats*"

macro_dir = INPUT_DIR / "macro"

csv_files = sorted(macro_dir.glob("*.csv"))
cols_keep = ['bundle_name', 'mean_length_mm', 'span_mm', 'curl', 'elongation', 'total_volume_mm3', '1st_quarter_volume_mm3',
             '2nd_and_3rd_quarter_volume_mm3', '4th_quarter_volume_mm3','total_surface_area_mm2', 'total_radius_of_end_regions_mm',
             'total_area_of_end_regions_mm2', 'irregularity', 'area_of_end_region_1_mm2', 'radius_of_end_region_1_mm',
             'volume_of_end_branches_1', 'area_of_end_region_2_mm2', 'radius_of_end_region_2_mm', 'volume_of_end_branches_2',
             'subject_id']
dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df = df[cols_keep].copy()
    df = df[df["subject_id"].isin(allowed_ids)]
    dfs.append(df)

macro_wide = pd.concat(dfs, ignore_index=True)

# long format: bundle_name + metric + mean + subject_id 
value_cols = [c for c in cols_keep if c not in ["bundle_name", "subject_id"]]

macro_long = macro_wide.melt(id_vars=["subject_id", "bundle_name"], value_vars=value_cols,
                             var_name="metric", value_name="mean").copy()

macro_long.rename(columns={"bundle_name": "bundle"}, inplace=True)

output_path = output_dir / "macro_data.csv"
macro_long.to_csv(output_path, index=False)

print(f"[SAVE] → {output_path} | shape: {macro_long.shape}")
macro_long.head()

In [ ]:
macro_long["subject_id"].nunique()

In [ ]:
# folder containing bundle qc data
# generated with: bash \
# 	unzip_files.sh \
# 	/cbica/projects/pennlinc_rbc/datasets/LINC_HBN/derivatives/QSIPREP-1-0-1_zipped \
# 	/cbica/projects/macedo_long_wm/data/qc \
# 	"qsiprep/sub-*/ses-1/dwi/sub-*desc-image_qc*"

qc_dir = INPUT_DIR / "qc"

qc_files = sorted(qc_dir.glob("sub-*desc-image_qc.tsv"))

qc_rows = []
for f in qc_files:
    m = re.search(r"(sub-[A-Za-z0-9]+)", f.name)
    subject_id = m.group(1)

    df_qc = pd.read_csv(f, sep="\t")
    val = df_qc["t1_neighbor_corr"].iloc[0]
    qc_rows.append({"subject_id": subject_id, "t1_neighbor_corr": val})

qc_df = pd.DataFrame(qc_rows)
qc_df["t1_neighbor_corr"] = pd.to_numeric(qc_df["t1_neighbor_corr"], errors="coerce")

output_path = output_dir / "qc_data.csv"
qc_df.to_csv(output_path, index=False)

print(f"[SAVE] → {output_path} | shape: {qc_df.shape}")
qc_df.head()

In [ ]:
qc_df["subject_id"].nunique()

In [ ]:
# combine macro + NODDI into one long dataframe
all_features_long = pd.concat([all_noddi, macro_long], ignore_index=True)

output_path = output_dir / "all_macro_noddi_long.csv"
all_features_long.to_csv(output_path, index=False)

print(f"[SAVE] → {output_path} | shape: {all_features_long.shape}")
all_features_long.head()

In [ ]:
all_features_long["subject_id"].nunique()

In [ ]:
participants = pd.read_csv(INPUT_DIR / "participants.tsv", sep="\t")
participants = participants[["participant_id", "Sex", "Age", "site"]]
participants.head(3)

In [ ]:
participants["participant_id"].nunique()

In [ ]:
df_long = all_features_long.copy()

# create a combined feature name: FEATURES_bundle_metric
df_long["feature_name"] = ("FEATURES_" + df_long["bundle"].astype(str) + "_" + df_long["metric"].astype(str))

# pivot to wide: one row per subject_id, one column per feature_name
df_wide = df_long.pivot(index="subject_id", columns="feature_name", values="mean")
df_wide = df_wide.reset_index()

# make subject id col consistent across dfs
general_ses = general_ses.copy()
general_ses["subject_id"] = "sub-" + general_ses["GUID"].astype(str)

# only keep subjects present in the features df
df_merged = df_wide.merge(general_ses[["subject_id", "GUID", "General_SES"]], on="subject_id", how="inner")

# merge in participant info
participants_sub = participants[["participant_id", "Sex", "Age", "site"]].copy()
df_with_demo = df_merged.merge(participants_sub, left_on="subject_id", right_on="participant_id", how="inner")

# reorder columns so participant_id, Sex, Age, site come first
feature_cols = [c for c in df_with_demo.columns if c.startswith("FEATURES_")]
first_cols = ["participant_id", "Sex", "Age", "site", "subject_id", "GUID", "General_SES"]
other_cols = [c for c in df_with_demo.columns if c not in first_cols + feature_cols]

df_final = df_with_demo[first_cols + other_cols + feature_cols].copy()
df_final = df_final.merge(qc_df, on="subject_id", how="left")

df_final.head()

In [ ]:
df_final["participant_id"].nunique()

In [ ]:
bundles_to_keep = [
    "Association_ArcuateFasciculusL",
    "Association_ArcuateFasciculusR",
    "Association_CingulumL",
    "Association_CingulumR",
    "Association_ExtremeCapsuleL",
    "Association_ExtremeCapsuleR",
    "Association_FrontalAslantTractL",
    "Association_FrontalAslantTractR",
    "Association_HippocampusAlveusL",
    "Association_HippocampusAlveusR",
    "Association_InferiorFrontoOccipitalFasciculusL",
    "Association_InferiorFrontoOccipitalFasciculusR",
    "Association_InferiorLongitudinalFasciculusL",
    "Association_InferiorLongitudinalFasciculusR",
    "Association_MiddleLongitudinalFasciculusL",
    "Association_MiddleLongitudinalFasciculusR",
    "Association_ParietalAslantTractL",
    "Association_ParietalAslantTractR",
    "Association_SuperiorLongitudinalFasciculusL",
    "Association_SuperiorLongitudinalFasciculusR",
    "Association_UncinateFasciculusL",
    "Association_UncinateFasciculusR",
    "Association_VerticalOccipitalFasciculusL",
    "Association_VerticalOccipitalFasciculusR",
    "Cerebellum_CerebellumL",
    "Cerebellum_CerebellumR",
    "Cerebellum_InferiorCerebellarPeduncleL",
    "Cerebellum_InferiorCerebellarPeduncleR",
    "Cerebellum_MiddleCerebellarPeduncle",
    "Cerebellum_SuperiorCerebellarPeduncle",
    "Cerebellum_Vermis",
    "Commissure_CorpusCallosum",
    "ProjectionBasalGanglia_AcousticRadiationL",
    "ProjectionBasalGanglia_AcousticRadiationR",
    "ProjectionBasalGanglia_AnsaLenticularisL",
    "ProjectionBasalGanglia_AnsaLenticularisR",
    "ProjectionBasalGanglia_AnsaSubthalamicaL",
    "ProjectionBasalGanglia_AnsaSubthalamicaR",
    "ProjectionBasalGanglia_CorticostriatalTractL",
    "ProjectionBasalGanglia_CorticostriatalTractR",
    "ProjectionBasalGanglia_FasciculusLenticularisL",
    "ProjectionBasalGanglia_FasciculusLenticularisR",
    "ProjectionBasalGanglia_FasciculusSubthalamicusL",
    "ProjectionBasalGanglia_FasciculusSubthalamicusR",
    "ProjectionBasalGanglia_FornixL",
    "ProjectionBasalGanglia_FornixR",
    "ProjectionBasalGanglia_OpticRadiationL",
    "ProjectionBasalGanglia_OpticRadiationR",
    "ProjectionBasalGanglia_ThalamicRadiationL",
    "ProjectionBasalGanglia_ThalamicRadiationR",
    "ProjectionBrainstem_CorticopontineTractL",
    "ProjectionBrainstem_CorticopontineTractR",
    "ProjectionBrainstem_CorticospinalTractL",
    "ProjectionBrainstem_CorticospinalTractR",
    "ProjectionBrainstem_MedialForebrainBundleL",
    "ProjectionBrainstem_MedialForebrainBundleR",
    "ProjectionBrainstem_MedialLemniscusL",
    "ProjectionBrainstem_MedialLemniscusR",
    "ProjectionBrainstem_NonDecussatingDentatorubrothalamicTractL",
    "ProjectionBrainstem_NonDecussatingDentatorubrothalamicTractR",
    "ProjectionBrainstem_ReticularTractL",
    "ProjectionBrainstem_ReticularTractR",
]

In [ ]:
def get_bundle_from_features(col):
    rest = col[len("FEATURES_"):]
    parts = rest.split("_")
    bundle = "_".join(parts[:2])
    return bundle

In [ ]:
# identify feature columns and map them to bundles
feature_cols = [c for c in df_final.columns if c.startswith("FEATURES_")]
keep_feature_cols = [c for c in feature_cols if get_bundle_from_features(c) in bundles_to_keep]
non_feature_cols = [c for c in df_final.columns if not c.startswith("FEATURES_")]
df_subset = df_final[non_feature_cols + keep_feature_cols].copy()

print(f"Total subjects before dropping NaNs: {df_subset.shape[0]}")
print(f"Total columns (including IDs, SES, demo, features): {df_subset.shape[1]}")

# subject-level missingness and complete cases
n_missing_per_subject = df_subset.isna().sum(axis=1)

print("\n=== Distribution of # of NaNs per subject (after bundle restriction) ===")
print(n_missing_per_subject.value_counts().sort_index())

n_subjects_total = df_subset.shape[0]
n_subjects_with_nan = (n_missing_per_subject > 0).sum()
n_subjects_complete = (n_missing_per_subject == 0).sum()

print(f"\nTotal subjects: {n_subjects_total}")
print(f"Subjects with ANY missing value: {n_subjects_with_nan}")
print(f"Subjects with NO missing values (complete cases): {n_subjects_complete}")

df_subset_complete = df_subset.dropna(axis=0, how="any").copy()

print(f"\nDropping subjects with any NaN after bundle restriction...")
print(f"Dropped {n_subjects_total - df_subset_complete.shape[0]} subjects")
print(f"Final sample size: {df_subset_complete.shape[0]} subjects")

In [ ]:
def summarize_hbn_demographics(df, age_col="Age", sex_col="Sex", sample_label="HBN"):
    df = df.copy()

    df[age_col] = pd.to_numeric(df[age_col], errors="coerce")
    df[sex_col] = pd.to_numeric(df[sex_col], errors="coerce")

    n = df.shape[0]
    age_m = df[age_col].mean()
    age_s = df[age_col].std(ddof=1)

    n_f = int((df[sex_col] == 1).sum())
    n_m = int((df[sex_col] == 0).sum())
    n_known = n_f + n_m

    pct_f = 100 * n_f / n_known if n_known > 0 else np.nan
    pct_m = 100 * n_m / n_known if n_known > 0 else np.nan

    return pd.DataFrame([{
        "Sample": sample_label,
        "n": n,
        "Age, mean (SD)": f"{age_m:.2f} ({age_s:.2f})",
        "Female, n (%)": f"{n_f} ({pct_f:.1f}%)",
        "Male, n (%)": f"{n_m} ({pct_m:.1f}%)"
    }])

hbn_demo_table = summarize_hbn_demographics(df_subset_complete, age_col="Age", sex_col="Sex")
hbn_demo_table

In [ ]:
df_subset_complete["participant_id"].nunique()

In [ ]:
feature_cols = [c for c in df_subset_complete.columns if c.startswith("FEATURES_")]

print("Total FEATURE columns:", len(feature_cols))
print(feature_cols[:10])

In [ ]:
output_path = output_dir / "final_hbn_dataset_12_23.csv"
df_subset_complete.to_csv(output_path)